# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank paper reports that its growth-prediction model, trained on about 96.6K clearly growing or declining pages, achieved about 90% accuracy on unseen pages from represented brands and about 75% on completely unseen brands.

**Methodology question:** Does the model still perform well on pages close to the growth/decline cutoff, or is most of its accuracy coming from pages with very obvious large changes?

This matters because restricting evaluation to clear positive and negative cases can make a classification problem easier than the borderline cases where a real decision is hardest. A useful audit would stratify performance by distance from the growth/decline threshold and check whether discrimination remains useful near that boundary.

### Finding 2 — 30-day momentum

The paper reports that its 30-day momentum model predicts whether a page will improve by more than 10% in the following month, with strong reported performance on both unseen pages from represented brands and completely unseen brands.

**Methodology question:** Were repeated observations from the same page or brand kept together during validation so that closely related observations could not appear in both training and test sets?

This matters because related observations can share persistent page- or brand-level behaviour. Keeping each repeated entity on only one side of a split tests whether performance transfers to genuinely independent entities rather than benefiting from information shared across closely related observations.


In [1]:
# Section 1 is a methodological reading exercise; no paper re-analysis is claimed here.
paper_questions = {
    "growth_prediction": "Does performance remain strong near the growth/decline cutoff?",
    "momentum_validation": "Were repeated page/brand observations kept together during validation?",
}
paper_questions


{'growth_prediction': 'Does performance remain strong near the growth/decline cutoff?',
 'momentum_validation': 'Were repeated page/brand observations kept together during validation?'}

## 2. My model under an honest split (before/after)

Assignment 6 already used client-grouped cross-validation, so I did not replace an actually used random split and pretend it was my historical Week-5 design. Instead, I reconstruct a **counterfactual naive before** using shuffled page-level 5-fold CV and compare it with the **honest after** using 5-fold `GroupKFold` by client.

Everything except the split is frozen: the same 1,800-page Assignment-6 training population, the same five March-only features, the same April targets, the same selected Random Forest hyperparameters, and the same ranking blend (`gamma=2`, `lambda=1`, with the Assignment-6 severity scale). This isolates the effect of allowing versus preventing the same client from appearing in both fit and validation folds.

The comparison reports classification ROC-AUC, regression RMSE, and ranking Precision@50. Because the decline target is common in this population, the observed decline base rate is printed alongside the metrics rather than interpreting Precision@50 in isolation.


In [2]:
# SECTION 2 — random page CV vs client-grouped CV using the frozen Assignment-6 models.
# The only intentional change is the split design.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.model_selection import GroupKFold, KFold

# ---------- warehouse access ----------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required to rebuild the exact Assignment-6 frame.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# ---------- rebuild the exact locked 2,520-page population ----------
march_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
client_tier_counts = (
    march_exposure.groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
eligible_clients = client_tier_counts[client_tier_counts.min(axis=1) >= 40].index.tolist()
balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(["client_hash_id", "exposure_tier"], observed=False, group_keys=False)
    .head(40).reset_index(drop=True)
)
balanced_keys = balanced_poc[["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("balanced_keys", balanced_keys)

march_features = con.sql(f"""
    WITH daily AS (
        SELECT f.client_hash_id, f.content_hash_id, f.report_date,
               DATE_DIFF('day', DATE '2026-03-01', f.report_date)::DOUBLE AS day_index,
               f.gsc_impressions::DOUBLE AS impressions,
               f.gsc_clicks::DOUBLE AS clicks,
               CASE WHEN f.gsc_avg_position >= 1
                    THEN f.gsc_avg_position::DOUBLE ELSE NULL END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
           MEDIAN(valid_position) AS median_position,
           REGR_SLOPE(valid_position, day_index)
               FILTER (WHERE valid_position IS NOT NULL) AS position_slope_per_day,
           QUANTILE_CONT(valid_position, 0.75) - QUANTILE_CONT(valid_position, 0.25)
               AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT d.client_hash_id, d.content_hash_id,
           DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')::DOUBLE
               AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr", "median_position", "position_slope_per_day",
    "position_iqr", "content_age_days"
]
feature_frame = (
    march_features.merge(age_feature, on=["client_hash_id", "content_hash_id"], how="inner")
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

con.register("model_keys", feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates())
march_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()
april_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()
target_frame = march_target.merge(april_target, on=["client_hash_id", "content_hash_id"], how="inner")
target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"] - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]
target_frame["future_decline"] = (target_frame["future_impression_change"] < 0).astype(int)

modeling_frame = (
    feature_frame.merge(
        target_frame[["client_hash_id", "content_hash_id", "future_impression_change", "future_decline"]],
        on=["client_hash_id", "content_hash_id"], how="inner"
    )
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

# Reuse the exact Assignment-5/6 training-client population; the six external clients remain untouched.
output_dir = Path("../outputs")
with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)
train_clients = set(split_manifest["train_clients"])
train_frame = (
    modeling_frame[modeling_frame["client_hash_id"].isin(train_clients)]
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert len(train_frame) == 1800
assert train_frame["client_hash_id"].nunique() == 15
assert train_frame[FINAL_FEATURES].notna().all().all()

X = train_frame[FINAL_FEATURES].copy()
y_cls = train_frame["future_decline"].astype(int).copy()
y_reg = train_frame["future_impression_change"].astype(float).copy()
groups = train_frame["client_hash_id"].copy()

# ---------- freeze the Assignment-6 selected models ----------
cls_model = RandomForestClassifier(
    n_estimators=400, random_state=42, n_jobs=-1,
    class_weight=None, max_depth=4, max_features="sqrt", min_samples_leaf=5,
)
reg_model = RandomForestRegressor(
    n_estimators=400, random_state=42, n_jobs=-1,
    max_depth=None, max_features="sqrt", min_samples_leaf=30,
)
RANK_GAMMA = 2.0
RANK_LAMBDA = 1.0
SEVERITY_SCALE = 0.47977155635777924

def evaluate_split(split_name, splits):
    rows = []
    for fold, (fit_idx, valid_idx) in enumerate(splits, start=1):
        X_fit, X_valid = X.iloc[fit_idx], X.iloc[valid_idx]
        yc_fit, yc_valid = y_cls.iloc[fit_idx], y_cls.iloc[valid_idx]
        yr_fit, yr_valid = y_reg.iloc[fit_idx], y_reg.iloc[valid_idx]

        c = clone(cls_model).fit(X_fit, yc_fit)
        r = clone(reg_model).fit(X_fit, yr_fit)
        p = c.predict_proba(X_valid)[:, 1]
        pred_r = r.predict(X_valid)

        auc = float(roc_auc_score(yc_valid, p))
        rmse = float(np.sqrt(mean_squared_error(yr_valid, pred_r)))

        severity = np.maximum(0.0, -pred_r)
        severity_norm = np.clip(severity / SEVERITY_SCALE, 0.0, 1.0)
        rank_score = np.power(np.clip(p, 1e-9, 1.0), RANK_GAMMA) * (
            1.0 + RANK_LAMBDA * severity_norm
        )
        rank_df = pd.DataFrame({"score": rank_score, "relevance": yc_valid.to_numpy()})
        rank_df = rank_df.sort_values("score", ascending=False)
        k = min(50, len(rank_df))
        p50 = float(rank_df.head(k)["relevance"].mean())

        fit_clients = set(groups.iloc[fit_idx])
        valid_clients = set(groups.iloc[valid_idx])
        rows.append({
            "split": split_name,
            "fold": fold,
            "validation_pages": int(len(valid_idx)),
            "validation_clients": int(len(valid_clients)),
            "client_overlap": int(len(fit_clients.intersection(valid_clients))),
            "decline_base_rate": float(yc_valid.mean()),
            "classification_roc_auc": auc,
            "regression_rmse": rmse,
            "ranking_precision_at_50": p50,
        })
    return pd.DataFrame(rows)

random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
grouped_cv = GroupKFold(n_splits=5)

random_rows = evaluate_split("random_page_cv", list(random_cv.split(X, y_cls)))
grouped_rows = evaluate_split("grouped_client_cv", list(grouped_cv.split(X, y_cls, groups=groups)))
fold_results = pd.concat([random_rows, grouped_rows], ignore_index=True)

summary = (
    fold_results.groupby("split", as_index=False)
    .agg(
        mean_client_overlap=("client_overlap", "mean"),
        mean_decline_base_rate=("decline_base_rate", "mean"),
        classification_roc_auc=("classification_roc_auc", "mean"),
        classification_roc_auc_sd=("classification_roc_auc", "std"),
        regression_rmse=("regression_rmse", "mean"),
        regression_rmse_sd=("regression_rmse", "std"),
        ranking_precision_at_50=("ranking_precision_at_50", "mean"),
        ranking_precision_at_50_sd=("ranking_precision_at_50", "std"),
    )
)

# The grouped rerun should reproduce the frozen Assignment-6 CV results.
grouped_summary = summary[summary["split"] == "grouped_client_cv"].iloc[0]
assert np.isclose(grouped_summary["classification_roc_auc"], 0.6650253727996942)
assert np.isclose(grouped_summary["regression_rmse"], 0.806274586196414)
assert np.isclose(grouped_summary["ranking_precision_at_50"], 0.8720000000000001)
assert (grouped_rows["client_overlap"] == 0).all()
assert (random_rows["client_overlap"] > 0).all()

receipt = {
    "comparison_role": "counterfactual naive random-page CV vs honest client-grouped CV",
    "population": {"pages": 1800, "clients": 15},
    "features": FINAL_FEATURES,
    "decline_base_rate": float(y_cls.mean()),
    "frozen_models": {
        "classifier": "RandomForestClassifier(max_depth=4, max_features='sqrt', min_samples_leaf=5, n_estimators=400)",
        "regressor": "RandomForestRegressor(max_features='sqrt', min_samples_leaf=30, n_estimators=400)",
        "ranking": {"gamma": RANK_GAMMA, "lambda": RANK_LAMBDA, "severity_scale": SEVERITY_SCALE},
    },
    "folds": fold_results.to_dict(orient="records"),
    "summary": summary.to_dict(orient="records"),
}
receipt_path = output_dir / "assignment7_split_audit.json"
with open(receipt_path, "w", encoding="utf-8") as fh:
    json.dump(receipt, fh, indent=2)

print("ASSIGNMENT 7 — HONEST SPLIT AUDIT")
print("Pages:", len(train_frame))
print("Clients:", train_frame["client_hash_id"].nunique())
print("Decline base rate:", round(float(y_cls.mean()), 4))
print("\nFold-level results:")
display(fold_results)
print("\nBefore/after summary:")
display(summary)
print("\nReceipt written:", receipt_path)


ASSIGNMENT 7 — HONEST SPLIT AUDIT
Pages: 1800
Clients: 15
Decline base rate: 0.7511

Fold-level results:


,split,fold,validation_pages,validation_clients,client_overlap,decline_base_rate,classification_roc_auc,regression_rmse,ranking_precision_at_50
0,random_page_cv,1,360,15,15,0.758333,0.745442,0.941418,1.00
1,random_page_cv,2,360,15,15,0.736111,0.743635,0.851227,0.94
2,random_page_cv,3,360,15,15,0.722222,0.730692,0.819649,0.94
3,random_page_cv,4,360,15,15,0.800000,0.679977,0.851611,0.92
4,random_page_cv,5,360,15,15,0.738889,0.723644,0.629250,0.94
5,grouped_client_cv,1,360,3,0,0.936111,0.697200,0.463367,1.00
6,grouped_client_cv,2,360,3,0,0.694444,0.528873,0.784814,0.70
7,grouped_client_cv,3,360,3,0,0.686111,0.689441,0.843647,0.86
8,grouped_client_cv,4,360,3,0,0.775000,0.808399,0.601016,1.00
9,grouped_client_cv,5,360,3,0,0.663889,0.601214,1.338529,0.80



Before/after summary:


,split,mean_client_overlap,mean_decline_base_rate,classification_roc_auc,classification_roc_auc_sd,regression_rmse,regression_rmse_sd,ranking_precision_at_50,ranking_precision_at_50_sd
0,grouped_client_cv,0.0,0.751111,0.665025,0.105826,0.806275,0.333493,0.872,0.130077
1,random_page_cv,15.0,0.751111,0.724678,0.026580,0.818631,0.115209,0.948,0.030332



Receipt written: ../outputs/assignment7_split_audit.json


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.